# Vaani-FastConformer-Hindi — IndicVoices Hindi Evaluation

**Model:** `ARTPARK-IISc/Vaani-FastConformer-Hindi`  
**Dataset:** `ai4bharat/IndicVoices` — Hindi / valid split  
**Platform:** Kaggle GPU T4  

### Features
- ✅ Batch inference with `tqdm` progress bar
- ✅ Checkpointing — saves every N batches; auto-resumes on re-run
- ✅ WER & CER via `jiwer`
- ✅ Per-sample predictions saved to `/kaggle/working/eval_results/`

### Before running
1. **Settings → Internet → On** (needed to pull model + dataset from HuggingFace)
2. Add your HF token as a Kaggle Secret:
   - *Add-ons → Secrets → + New Secret*
   - **Name:** `HF_TOKEN`  **Value:** your token from https://huggingface.co/settings/tokens
3. Set accelerator: *Session options → Accelerator → GPU T4 x2*


## Cell 1 — Install Dependencies

In [4]:
%%capture
# NeMo + ASR extras (~4-5 min first run; cached on re-run)
!pip install Cython
!pip install 'nemo_toolkit[asr]'
!pip install 'datasets>=2.14' 'huggingface_hub>=0.20' jiwer soundfile librosa
print('Dependencies installed ✅')


## Cell 2 — HuggingFace Authentication

In [5]:
import os
from huggingface_hub import login

# Pull token from Kaggle Secret (set via Add-ons → Secrets)
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login successful ✅')


HuggingFace login successful ✅


## Cell 3 — Configuration

In [6]:
import os

# ── Model & Dataset ──────────────────────────────────────────────────────
MODEL_ID        = 'ARTPARK-IISc/Vaani-FastConformer-Hindi'
DATASET_ID      = 'ai4bharat/IndicVoices'
LANG_CONFIG     = 'hindi'    # IndicVoices config name
# NOTE: IndicVoices Hindi on HuggingFace only exposes a 'valid' split.
#       Use it as the test/evaluation set.
SPLIT           = 'valid'
TARGET_SR       = 16_000    # NeMo expects 16 kHz

# ── Inference ────────────────────────────────────────────────────────────
BATCH_SIZE      = 8          # lower to 4 if OOM
MAX_SAMPLES     = None       # e.g. 200 for a quick smoke-test; None = full set

# ── Checkpointing ────────────────────────────────────────────────────────
CHECKPOINT_EVERY = 50        # save a checkpoint every N batches
OUTPUT_DIR       = '/kaggle/working/eval_results'
CHECKPOINT_FILE  = os.path.join(OUTPUT_DIR, 'checkpoint.json')
PREDICTIONS_FILE = os.path.join(OUTPUT_DIR, 'predictions.jsonl')
METRICS_FILE     = os.path.join(OUTPUT_DIR, 'metrics.json')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready ✅')
print(f'  Output dir      : {OUTPUT_DIR}')
print(f'  Checkpoint every: {CHECKPOINT_EVERY} batches')


Config ready ✅
  Output dir      : /kaggle/working/eval_results
  Checkpoint every: 50 batches


## Cell 4 — Load Model

In [7]:
import torch
import nemo.collections.asr as nemo_asr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    # T4 supports float16; force float32 for numerical safety with NeMo
    torch.set_default_dtype(torch.float32)
    torch.backends.cuda.matmul.allow_tf32 = False

# Load on CPU first to avoid CUDA dtype assertion during weight loading
print('Loading model on CPU ...')
model = nemo_asr.models.ASRModel.from_pretrained(
    model_name=MODEL_ID,
    map_location='cpu',
)

# Cast all parameters to float32 (checkpoint may contain bf16 tensors)
model = model.float()

# Move to GPU
model = model.to(device)
model.eval()
print(f'Model loaded on {device} as float32 ✅')


[NeMo W 2026-06-15 10:04:34 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-06-15 10:04:38 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-06-15 10:04:38 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-06-15 10:04:38 nemo_logging:364] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.

Device : cuda
GPU    : Tesla T4
Loading model on CPU ...


Vaani-FastConformer-Hindi.nemo:   0%|          | 0.00/1.75G [00:00<?, ?B/s]

[NeMo I 2026-06-15 10:04:54 mixins:184] Tokenizer SentencePieceTokenizer initialized with 800 tokens


[NeMo W 2026-06-15 10:04:55 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetuningDataset/manifest_train_hindi_cleaned.json
    sample_rate: 16000
    use_start_end_token: false
    batch_size: 16
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 40
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-06-15 10:04:55 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath:
    - /home/sujith/asrTraining/finetun

[NeMo I 2026-06-15 10:04:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 10:04:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 10:04:58 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-06-15 10:05:00 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--ARTPARK-IISc--Vaani-FastConformer-Hindi/snapshots/9fa335764565e4407838066d904fa101282744f4/Vaani-FastConformer-Hindi.nemo.
Model loaded on cuda as float32 ✅


## Cell 5 — Load IndicVoices Hindi Dataset

In [5]:
from datasets import load_dataset

print(f'Loading {DATASET_ID} [{LANG_CONFIG}] split={SPLIT} ...')
dataset = load_dataset(DATASET_ID, LANG_CONFIG, split=SPLIT, trust_remote_code=True)

if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

print(f'Total samples : {len(dataset)}')
print(f'Columns       : {dataset.column_names}')

# Inspect first sample
s0 = dataset[0]
print('\nFirst sample preview:')
for k, v in s0.items():
    if k != 'audio':
        print(f'  {k}: {str(v)[:120]}')
    else:
        print(f'  audio: array shape={len(v["array"])}, sr={v["sampling_rate"]}')


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/IndicVoices' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ai4bharat/IndicVoices [hindi] split=valid ...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/82 [00:00<?, ?it/s]

...

hindi/train-00080-of-00082.parquet:   0%|          | 0.00/605M [00:00<?, ?B/s]

hindi/train-00081-of-00082.parquet:   0%|          | 0.00/578M [00:00<?, ?B/s]

Generating valid split:   0%|          | 0/5530 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/445160 [00:00<?, ? examples/s]

Total samples : 5530
Columns       : ['audio_filepath', 'text', 'duration', 'lang', 'samples', 'verbatim', 'normalized', 'speaker_id', 'scenario', 'task_name', 'gender', 'age_group', 'job_type', 'qualification', 'area', 'district', 'state', 'occupation', 'verification_report', 'unsanitized_verbatim', 'unsanitized_normalized']

First sample preview:
  audio_filepath: <datasets.features._torchcodec.AudioDecoder object at 0x7d4b34b765a0>
  text: स्थानीय कला रूप या शिल्प होसंगाबाद जिले के लिए एक
  duration: 5.729
  lang: hi
  samples: 91664
  verbatim: इस्थानीय कला रूप या सिल्प होसंगाबाद जिले के लिए एक
  normalized: स्थानीय कला रूप या शिल्प होसंगाबाद जिले के लिए एक
  speaker_id: S4259869900354210
  scenario: Extempore
  task_name: District Specific
  gender: Male
  age_group: 30-45
  job_type: Blue Collar
  qualification: Post Grad + PhD
  area: Rural
  district: Hoshangabad
  state: Madhya Pradesh
  occupation: Fermar
  verification_report: {'sst': False, 'comments': 'Somehow this extempo

## Cell 6 — Helper Functions

In [8]:
import numpy as np
import librosa
import soundfile as sf
import tempfile, uuid, json

# ── Audio ────────────────────────────────────────────────────────────────
def get_audio_array(sample) -> np.ndarray:
    audio = sample['audio']
    arr   = np.array(audio['array'], dtype=np.float32)
    sr    = audio['sampling_rate']
    if sr != TARGET_SR:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
    return arr

# ── Reference transcript ─────────────────────────────────────────────────
TRANSCRIPT_KEYS = ('text', 'transcript', 'sentence', 'normalized_text')

def get_reference(sample) -> str:
    for key in TRANSCRIPT_KEYS:
        if key in sample and sample[key]:
            return sample[key].strip()
    raise KeyError(f'No transcript column found. Available: {list(sample.keys())}')

# ── Batch transcription ──────────────────────────────────────────────────
def transcribe_batch(samples):
    tmp_paths = []
    for s in samples:
        arr  = get_audio_array(s)
        path = os.path.join(tempfile.gettempdir(), f'{uuid.uuid4().hex}.wav')
        sf.write(path, arr, TARGET_SR)
        tmp_paths.append(path)
    try:
        hyps = model.transcribe(tmp_paths, batch_size=len(tmp_paths))
        # NeMo may return strings or Hypothesis objects
        return [h.text if hasattr(h, 'text') else str(h) for h in hyps]
    finally:
        for p in tmp_paths:
            try: os.remove(p)
            except OSError: pass

# ── Checkpoint I/O ───────────────────────────────────────────────────────
def save_checkpoint(global_batch_idx: int, num_done: int):
    """Persist progress so the run can be resumed after interruption."""
    ckpt = {'next_batch': global_batch_idx + 1, 'num_done': num_done}
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(ckpt, f)

def load_checkpoint():
    """Returns (start_batch_index, already_done_count)."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            ckpt = json.load(f)
        print(f'🔁 Resuming from batch {ckpt["next_batch"]}  '
              f'({ckpt["num_done"]} samples already done)')
        return ckpt['next_batch'], ckpt['num_done']
    return 0, 0

def load_existing_predictions():
    """Read predictions already written to the JSONL file."""
    results = []
    if os.path.exists(PREDICTIONS_FILE):
        with open(PREDICTIONS_FILE) as f:
            for line in f:
                line = line.strip()
                if line:
                    results.append(json.loads(line))
    return results

print('Helpers defined ✅')


Helpers defined ✅


## Cell 7 — Run Inference with Checkpointing

Re-run this cell at any time — it automatically resumes from the last checkpoint.

In [21]:
import os, json

# Reset checkpoint and predictions
for f in [CHECKPOINT_FILE, PREDICTIONS_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f'Deleted: {f}')

sample_log = []
print('Ready for a clean run ✅')

Deleted: /kaggle/working/eval_results/checkpoint.json
Deleted: /kaggle/working/eval_results/predictions.jsonl
Ready for a clean run ✅


In [22]:
from tqdm.notebook import tqdm
from itertools import islice

# ── Resume or start fresh ────────────────────────────────────────────────────
ckpt_data     = json.load(open(CHECKPOINT_FILE)) if os.path.exists(CHECKPOINT_FILE) else {}
skip_samples  = ckpt_data.get('num_done', 0)
sample_log    = load_existing_predictions()

if skip_samples:
    print(f'🔁 Resuming — skipping first {skip_samples} samples already done')
else:
    print('Starting fresh')
print()

def take_batch(it, n):
    return list(islice(it, n))

stream = load_dataset(
    DATASET_ID, LANG_CONFIG, split=SPLIT,
    streaming=True, trust_remote_code=True,
)
if MAX_SAMPLES:
    stream = stream.take(MAX_SAMPLES)

stream_iter = iter(stream.skip(skip_samples))
pred_file   = open(PREDICTIONS_FILE, 'a', encoding='utf-8')
batch_num   = 0
skipped     = 0
total       = (MAX_SAMPLES - skip_samples) if MAX_SAMPLES else None

try:
    with tqdm(desc='Samples', unit='sample', total=total) as pbar:
        while True:
            batch = take_batch(stream_iter, BATCH_SIZE)
            if not batch:
                break

            # ── Try full batch first; fall back to one-by-one on decoder error ──
            try:
                preds = transcribe_batch(batch)
                pairs = list(zip(batch, preds))
            except RuntimeError as e:
                if 'decoder' not in str(e).lower() and 'invalid data' not in str(e).lower():
                    raise          # re-raise unrelated errors
                tqdm.write(f'⚠️  Batch {batch_num} decoder error — retrying sample-by-sample')
                pairs = []
                for s in batch:
                    try:
                        pred = transcribe_batch([s])
                        pairs.append((s, pred[0]))
                    except RuntimeError:
                        skipped += 1
                        tqdm.write(f'   ↳ skipped 1 bad sample (total skipped: {skipped})')

            for s, pred in pairs:
                ref   = get_reference(s)
                entry = {'reference': ref, 'hypothesis': pred}
                sample_log.append(entry)
                pred_file.write(json.dumps(entry, ensure_ascii=False) + '\n')

            pred_file.flush()
            pbar.update(len(pairs))
            pbar.set_postfix(skipped=skipped)
            batch_num += 1

            if batch_num % CHECKPOINT_EVERY == 0:
                save_checkpoint(batch_num - 1, len(sample_log))
                tqdm.write(f'💾 Checkpoint @ batch {batch_num}  '
                           f'({len(sample_log)} done, {skipped} skipped)')

    save_checkpoint(batch_num - 1, len(sample_log))
    print(f'\n✅ Inference complete — {len(sample_log)} processed, {skipped} skipped')

finally:
    pred_file.close()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/IndicVoices' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/IndicVoices' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Starting fresh



Resolving data files:   0%|          | 0/112 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/82 [00:00<?, ?it/s]

Samples: 0sample [00:00, ?sample/s]

...

💾 Checkpoint @ batch 600  (4799 done, 1 skipped)



Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.83it/s]
[NeMo W 2026-06-15 11:09:54 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 11:09:54 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  2.43it/s]
[NeMo W 2026-06-15 11:09:55 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 11:09:55 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cau

💾 Checkpoint @ batch 650  (5199 done, 1 skipped)



Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  1.62it/s]
[NeMo W 2026-06-15 11:10:27 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 11:10:27 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:00,  5.52it/s]
[NeMo W 2026-06-15 11:10:27 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-06-15 11:10:27 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cau


✅ Inference complete — 5529 processed, 1 skipped


## Cell 8 — Compute WER & CER

In [23]:
from jiwer import wer, cer

references  = [e['reference']  for e in sample_log]
hypotheses  = [e['hypothesis'] for e in sample_log]

word_error_rate = wer(references, hypotheses)
char_error_rate = cer(references, hypotheses)

print('=' * 55)
print(f'  Model   : {MODEL_ID}')
print(f'  Dataset : {DATASET_ID} / {LANG_CONFIG} / {SPLIT}')
print(f'  Samples : {len(references)}')
print(f'  WER     : {word_error_rate * 100:.2f}%')
print(f'  CER     : {char_error_rate * 100:.2f}%')
print('=' * 55)


  Model   : ARTPARK-IISc/Vaani-FastConformer-Hindi
  Dataset : ai4bharat/IndicVoices / hindi / valid
  Samples : 5529
  WER     : 15.11%
  CER     : 7.09%


## Cell 9 — Per-sample Error Analysis

In [24]:
import pandas as pd
from jiwer import wer as single_wer

rows = []
for e in sample_log:
    try:
        s_wer = single_wer(e['reference'], e['hypothesis'])
    except Exception:
        s_wer = float('nan')
    rows.append({**e, 'sample_wer': round(s_wer, 4)})

df        = pd.DataFrame(rows)
df_sorted = df.sort_values('sample_wer', ascending=False)

print('\n── Top-10 worst predictions ──')
display(df_sorted.head(10))

print('\n── Top-10 best predictions ──')
display(df_sorted.tail(10).iloc[::-1])



── Top-10 worst predictions ──


,reference,hypothesis,sample_wer
4943,<unintelligible>,क्योंकि डबल स्टैचू होता है एक यही कम स्पाइस हो...,14.0
4935,<unintelligible>,मैं जाना चाहता हूँ अलमारी जनमन मिल रहा है,9.0
4240,प्रियांस <unintelligible>,राम से कचौ रहे हैं,2.5
108,<unintelligible>,हाँ जी,2.0
5459,काहे,आएँ हैं,2.0
767,आजकल,आज कल,2.0
968,जी,जी हाँ हाँ,2.0
4955,<unintelligible>,जी जी,2.0
4950,<unintelligible>,ठीक है,2.0
2450,समस्तीपुर,नमस्ते सर,2.0



── Top-10 best predictions ──


,reference,hypothesis,sample_wer
2962,ठीक है ठीक है,ठीक है ठीक है,0.0
4242,दो चार दिन लगेंगे,दो चार दिन लगेंगे,0.0
4243,हाँ,हाँ,0.0
457,लकड़ियों का उपयोग किया जाता था,लकड़ियों का उपयोग किया जाता था,0.0
458,जिनमें धुआँ होता था परंतु आज कल,जिनमें धुआँ होता था परंतु आज कल,0.0
4244,हाँ,हाँ,0.0
4245,हाँ पूछो,हाँ पूछो,0.0
4246,ग्यारह खिलाड़ी होते हैं,ग्यारह खिलाड़ी होते हैं,0.0
4247,हाँ,हाँ,0.0
4248,इसमें कैसे करना पड़ता है,इसमें कैसे करना पड़ता है,0.0


## Cell 10 — Save Metrics & Output Files

In [25]:
import json

metrics = {
    'model'      : MODEL_ID,
    'dataset'    : DATASET_ID,
    'config'     : LANG_CONFIG,
    'split'      : SPLIT,
    'num_samples': len(references),
    'WER'        : round(word_error_rate, 6),
    'CER'        : round(char_error_rate, 6),
}

with open(METRICS_FILE, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

csv_path = os.path.join(OUTPUT_DIR, 'predictions.csv')
df.to_csv(csv_path, index=False)

print(f'Saved: {METRICS_FILE}')
print(f'Saved: {PREDICTIONS_FILE}')
print(f'Saved: {csv_path}')
print()
print('All output files → /kaggle/working/eval_results/')
print('Download via the Output panel on the right →')
print()
print(json.dumps(metrics, indent=2))


Saved: /kaggle/working/eval_results/metrics.json
Saved: /kaggle/working/eval_results/predictions.jsonl
Saved: /kaggle/working/eval_results/predictions.csv

All output files → /kaggle/working/eval_results/
Download via the Output panel on the right →

{
  "model": "ARTPARK-IISc/Vaani-FastConformer-Hindi",
  "dataset": "ai4bharat/IndicVoices",
  "config": "hindi",
  "split": "valid",
  "num_samples": 5529,
  "WER": 0.151128,
  "CER": 0.070917
}
